# Chapter 3: Data Understanding & Preparation
### Exploratory Data Analysis (EDA) on Real Mobile Video
This notebook runs our local YOLOv5 model on a raw mobile video upload to generate real-world statistics for our thesis. We will extract bounding box **Aspect Ratios** and measure **Frame Sharpness (Laplacian Variance)**.

In [ ]:
import sys
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

# Setup paths (Adjust if running from a different directory)
BACKEND_DIR = Path(os.getcwd()).parent
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))
    
# Import our YOLOv5 Service
from app.services.yolov5_service import YOLOv5Service

# Set the video path
VIDEO_PATH = BACKEND_DIR / "uploads" / "f4603c58-571f-4082-b29a-1b4c67529cc7.mp4"
print(f"Target Video: {VIDEO_PATH.name}")

### 1. Initialize Local YOLOv5 Model

In [ ]:
print("Loading local YOLOv5s model...")
yolo_service = YOLOv5Service()
print("Model loaded successfully!")

### 2. Video Frame Extraction & Inference
We process every Nth frame of the video, running YOLOv5 to find appliances and calculate their bounding box aspect ratios.

In [ ]:
cap = cv2.VideoCapture(str(VIDEO_PATH))
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = cap.get(cv2.CAP_PROP_FPS)

print(f"Video contains {total_frames} frames at {fps:.1f} FPS (Duration: {total_frames/fps:.1f}s)")

aspect_ratios = {}
sharpness_scores = []
frame_indices = []

# Process every 4th frame (to save time in the notebook)
frame_interval = 4 

for frame_idx in range(0, total_frames, frame_interval):
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
    ret, frame = cap.read()
    if not ret:
        continue
        
    # Calculate Sharpness (Laplacian Variance)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    sharpness = cv2.Laplacian(gray, cv2.CV_64F).var()
    sharpness_scores.append(sharpness)
    frame_indices.append(frame_idx)
        
    # Convert for YOLO
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb_frame)
    
    # Run Inference
    detections = yolo_service.detect(pil_image)
    
    # Save a screenshot if we find a high-confidence refrigerator
    if frame_idx == 84 and detections: # Frame 84 is known to be sharp
        sample_screenshot = yolo_service.draw_detections(pil_image, detections)
        
    for det in detections:
        cls_name = det['class']
        x1, y1, x2, y2 = det['bbox']
        width = x2 - x1
        height = y2 - y1
        
        if height > 0:
            ar = width / height
            if cls_name not in aspect_ratios:
                aspect_ratios[cls_name] = []
            aspect_ratios[cls_name].append(ar)

cap.release()
print("Finished processing video.")

### 3. Aspect Ratio Distribution
Visualizing the physical form factors of detected appliances to justify our anchor box sizing.

In [ ]:
plt.figure(figsize=(12, 6))
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']

# Filter to just our target forensic classes (or anything we found)
target_classes = ['refrigerator', 'microwave', 'oven', 'tv', 'laptop']

plot_idx = 0
for cls_name, ars in aspect_ratios.items():
    if cls_name in target_classes:
        plt.hist(ars, bins=15, alpha=0.7, label=f"{cls_name.capitalize()} (n={len(ars)})", color=colors[plot_idx % len(colors)])
        plot_idx += 1

plt.title("Real Bounding Box Aspect Ratios (from Video Extracted Data)", fontsize=14)
plt.xlabel("Aspect Ratio (Width/Height)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

### 4. Frame Sharpness over Time (Laplacian Variance)
Why temporal filtering is necessary: Most frames in a panning video are severely motion-blurred.

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(frame_indices, sharpness_scores, label="Laplacian Variance (S)", color="#2ca02c")
plt.axhline(y=50, color='r', linestyle='--', label="Blur Threshold (S=50)")

# Highlight the best frame
best_idx = np.argmax(sharpness_scores)
best_frame = frame_indices[best_idx]
best_score = sharpness_scores[best_idx]

plt.scatter([best_frame], [best_score], color='blue', s=100, zorder=5, label=f"Hero Frame {best_frame} (S={best_score:.1f})")

plt.title("Frame Sharpness (Motion Blur) over Video Timeline", fontsize=14)
plt.xlabel("Frame Index", fontsize=12)
plt.ylabel("Sharpness Score (S)", fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.fill_between(frame_indices, 0, 50, color='red', alpha=0.1)
plt.show()

### 5. Hero Frame Raw Detection Result
Showing the highest quality frame identified by the pipeline with its local YOLOv5 bounding boxes.

In [ ]:
# Display the screenshot we captured earlier
plt.figure(figsize=(12, 8))
plt.imshow(sample_screenshot)
plt.axis('off')
plt.title(f"Hero Frame YOLOv5 Detection (Frame 84)", fontsize=14)
plt.show()